# 01 · Exploratory Data Analysis

Preliminary analysis of the company documents and the Y Combinator dataset: word counts and TF-IDF cosine similarity between company material and the evaluation dimensions.

> **Private data.** The two document samples analysed below (`Data sample 1`, `Data sample 2`) are confidential Azimut Zero material and are **not** included in this repository. Point the paths at your local copies to run those cells. The final section uses the public YC dataset in `data/raw/` and runs as-is.

Text helpers live in `company_assessment.eda`. Install the document readers with `pip install -e .[eda]`.

In [ ]:
import pandas as pd

from company_assessment import config
from company_assessment.eda import (
    read_pdf_file,
    read_excel_file,
    read_word_file,
    clean_text,
    count_words,
    top_10_frequent_words,
    remove_stop_words,
    calculate_cosine_similarity,
)

## Document samples (private)

Set these paths to your local copies of the confidential samples.

In [ ]:
# Data sample 1
data_1_pdf = 'Data/Data sample 1/ai deck.pdf'
data_1_excel = 'Data/Data sample 1/data sample 1.xlsx'

# Data sample 2
data_2_pdf = 'Data/Data sample 2/20240102_AzimutZer0_Slides.pdf'
data_2_pdf_2 = 'Data/Data sample 2/Q10. Models Finetuning Process.pdf'
data_2_pdf_3 = 'Data/Data sample 2/Q6. Quality Assurance overview.pdf'
data_2_excel = 'Data/Data sample 2/data sample 2.xlsx'
data_2_doc_1 = 'Data/Data sample 2/Q17. 20240102 Update to team_ch6.docx'
data_2_doc_2 = 'Data/Data sample 2/Q9. Update to team_ch5.docx'

In [ ]:
# Data sample 1
data_1_pdf_data = read_pdf_file(data_1_pdf)
data_1_excel_data = read_excel_file(data_1_excel)

# Data sample 2
data_2_pdf_data = read_pdf_file(data_2_pdf)
data_2_pdf_2_data = read_pdf_file(data_2_pdf_2)
data_2_pdf_3_data = read_pdf_file(data_2_pdf_3)
data_2_excel_data = read_excel_file(data_2_excel)
data_2_doc_1_data = read_word_file(data_2_doc_1)
data_2_doc_2_data = read_word_file(data_2_doc_2)

Drop the free-text `Question` / `Comment` columns from the spreadsheets.

In [ ]:
data_1_excel_data.drop(['Question', 'Comment'], axis=1, inplace=True)
data_2_excel_data.drop(['Question', 'Comment'], axis=1, inplace=True)

### Clean the extracted text

In [ ]:
data_1_pdf_data = clean_text(data_1_pdf_data)
data_2_pdf_data = clean_text(data_2_pdf_data)
data_2_pdf_2_data = clean_text(data_2_pdf_2_data)
data_2_pdf_3_data = clean_text(data_2_pdf_3_data)
data_2_doc_1_data = clean_text(data_2_doc_1_data)
data_2_doc_2_data = clean_text(data_2_doc_2_data)

## Word counts

In [ ]:
data_1_excel_text = ' '.join(data_1_excel_data.iloc[:, 0].astype(str))
data_2_excel_text = ' '.join(data_2_excel_data.iloc[:, 0].astype(str))

sample_1_words = count_words(data_1_pdf_data) + count_words(data_1_excel_text)
sample_2_words = (
    count_words(data_2_pdf_data)
    + count_words(data_2_pdf_2_data)
    + count_words(data_2_pdf_3_data)
    + count_words(data_2_excel_text)
    + count_words(data_2_doc_1_data)
    + count_words(data_2_doc_2_data)
)
print('Sample 1 words:', sample_1_words)
print('Sample 2 words:', sample_2_words)

## Most frequent words

In [ ]:
joint_text_1 = ' '.join([data_1_pdf_data, data_1_excel_text])
joint_text_2 = ' '.join([
    data_2_pdf_data, data_2_pdf_2_data, data_2_pdf_3_data,
    data_2_excel_text, data_2_doc_1_data, data_2_doc_2_data,
])

print('Sample 1:', top_10_frequent_words(joint_text_1, remove_stopwords=True))
print('Sample 2:', top_10_frequent_words(joint_text_2, remove_stopwords=True))

## Cosine similarity to the evaluation dimensions

In [ ]:
list_of_dimensions = ['Dataset size & quality', 'AI product roadmap', 'AI & data strategy']

print('Sample 1:', calculate_cosine_similarity(remove_stop_words(joint_text_1), list_of_dimensions))
print('Sample 2:', calculate_cosine_similarity(remove_stop_words(joint_text_2), list_of_dimensions))

## Public YC dataset

The AI-company filtering used to build the evaluation set (see notebook 02 for the full pipeline).

In [ ]:
import ast

ycd = pd.read_csv(config.YC_COMPANIES_CSV)
ycd['tags'] = ycd['tags'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else x
)
ai_companies = ycd[ycd['tags'].apply(
    lambda tags: 'Artificial Intelligence' in tags if isinstance(tags, list) else False
)]
print(f'{len(ai_companies)} AI-tagged companies')
ai_companies.head()